In [1]:
import ee
import pandas as pd

In [2]:
# 1. Authenticate and initialize GEE
# ==========================================
ee.Authenticate()

True

In [3]:
ee.Initialize(project="sundarbans-climate-analysis")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [4]:
# 2. Define study area
# Approximate Sundarbans coastal region
# ==========================================
roi = ee.Geometry.Rectangle(
    [88.5, 20.5, 90.5, 22.0]
)

In [5]:
# 3. Load HYCOM data
# ==========================================
collection = (
    ee.ImageCollection("HYCOM/sea_surface_elevation")
    .filterDate("2000-01-01", "2024-09-06")
    .select("surface_elevation")
)

print("Available image count:", collection.size().getInfo())

Available image count: 24382


In [6]:
# Convert catalog values to meters using scale factor 0.001
def convert_to_meters(image):
    return (
        image.multiply(0.001)
        .rename("elevation_m")
        .copyProperties(image, ["system:time_start"])
    )

elevation = collection.map(convert_to_meters)


In [10]:
# 4. Generate all months from 2000 to 2024
# ==========================================
start_date = ee.Date("2000-01-01")
end_date = ee.Date("2024-09-06")

total_months = end_date.difference(
    start_date, "month"
).ceil()

month_indices = ee.List.sequence(
    0, total_months.subtract(1)
)



In [11]:
# 5. Calculate monthly mean
# ==========================================
def calculate_monthly_mean(index):
    month_start = start_date.advance(index, "month")
    month_end = month_start.advance(1, "month")

    monthly_collection = elevation.filterDate(
        month_start, month_end
    )

    image_count = monthly_collection.size()

    monthly_image = monthly_collection.mean()

    stats = monthly_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=9000,
        maxPixels=1e8,
        bestEffort=True
    )

    return ee.Feature(None, {
        "Year": month_start.get("year"),
        "Month": month_start.get("month"),
        "Date": month_start.format("YYYY-MM"),
        "Mean_Elevation_Anomaly_m": stats.get("elevation_m"),
        "Available_Image_Count": image_count
    })

monthly_data = ee.FeatureCollection(
    month_indices.map(calculate_monthly_mean)
)

# Remove months without observations
monthly_data = monthly_data.filter(
    ee.Filter.gt("Available_Image_Count", 0)
).filter(
    ee.Filter.notNull(["Mean_Elevation_Anomaly_m"])
)


In [12]:
# 6. Retrieve results in Python
# ==========================================
result = monthly_data.getInfo()

rows = [
    feature["properties"]
    for feature in result["features"]
]

df = pd.DataFrame(rows)

df = df.sort_values(
    ["Year", "Month"]
).reset_index(drop=True)


In [13]:
# 7. Export to Excel
# ==========================================
output_file = "HYCOM_Sundarbans_Monthly_2000_2024.xlsx"

with pd.ExcelWriter(
    output_file, engine="openpyxl"
) as writer:
    df.to_excel(
        writer,
        sheet_name="Monthly_Data",
        index=False
    )

print("\nMonthly Excel file created successfully!")
print("File:", output_file)
print("Number of monthly records:", len(df))
print(df.head(12).to_string(index=False))



Monthly Excel file created successfully!
File: HYCOM_Sundarbans_Monthly_2000_2024.xlsx
Number of monthly records: 297
 Available_Image_Count    Date  Mean_Elevation_Anomaly_m  Month  Year
                    31 2000-01                  0.407380      1  2000
                    29 2000-02                  0.376481      2  2000
                    31 2000-03                  0.445538      3  2000
                    30 2000-04                  0.532423      4  2000
                    28 2000-05                  0.621180      5  2000
                    30 2000-06                  0.645706      6  2000
                    31 2000-07                  0.648830      7  2000
                    31 2000-08                  0.642987      8  2000
                    30 2000-09                  0.573857      9  2000
                    31 2000-10                  0.613979     10  2000
                    30 2000-11                  0.603771     11  2000
                    31 2000-12           